In [1]:
import polars as pl 
from dbconfig import engine
print('Environment Ready!')

Environment Ready!


In [16]:
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(100)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(40000)

polars.config.Config

In [2]:
pl.read_database(
        query = """
        select table_name, column_name
        from information_schema.columns
        where table_schema = 'raw'
        order by table_name, ordinal_position
        """, connection = engine)

table_name,column_name
str,str
"""category_translation""","""product_category_name"""
"""category_translation""","""product_category_name_english"""
"""customers""","""customer_id"""
"""customers""","""customer_unique_id"""
"""customers""","""customer_zip_code_prefix"""
…,…
"""reviews""","""review_answer_timestamp"""
"""sellers""","""seller_id"""
"""sellers""","""seller_zip_code_prefix"""


In [19]:
pl.read_database(
        query = """
        select string_agg(format(
            'select %L as table_name,
            count(*) as total_rows 
            from raw.%I', table_name, table_name),E'\nunion all\n')
        from(
            select distinct table_name
            from information_schema.columns
            where table_schema = 'raw') t;
        """,connection = engine)

string_agg
str
"""select 'order_items' as table_name, count(*) as total_rows from raw.order_items union all select 'geolocation' as table_name, count(*) as total_rows from raw.geolocation union all select 'sellers' as table_name, count(*) as total_rows from raw.sellers union all select 'orders' as table_name, count(*) as total_ro…"


In [3]:
very_small_query = pl.read_database(
        query = """
        select string_agg( format(
            'select %L as table_name,
            %L as column_name,
            count(*) as total_rows,
            count(%s) as non_nulls,
            count(*) - count(%s) as nulls,
            round(100.00 * (count(*) - count(%s)) / count(*), 2) as null_pct
            from raw.%I', table_name, column_name, column_name, column_name, column_name, table_name), E'\nunion all\n')
        from(
            select table_name,
            column_name
            from information_schema.columns 
            where table_schema = 'raw'
            order by table_name, ordinal_position)
        """, connection = engine)

In [4]:
query = very_small_query.item()

In [5]:
null_audit = pl.read_database(
        query = query,
        connection = engine)

In [6]:
null_audit.head()

table_name,column_name,total_rows,non_nulls,nulls,null_pct
str,str,i64,i64,i64,"decimal[38,2]"
"""category_translation""","""product_category_name""",71,71,0,0.00
"""category_translation""","""product_category_name_english""",71,71,0,0.00
"""sellers""","""seller_state""",3095,3095,0,0.00
"""sellers""","""seller_city""",3095,3095,0,0.00
"""sellers""","""seller_zip_code_prefix""",3095,3095,0,0.00


In [30]:
null_audit.filter(
        pl.col("nulls") > 0
        ).sort(
                "null_pct", descending = True)

table_name,column_name,total_rows,non_nulls,nulls,null_pct
str,str,i64,i64,i64,"decimal[38,2]"
"""reviews""","""review_comment_title""",99224,11568,87656,88.34
"""reviews""","""review_comment_message""",99224,40977,58247,58.70
"""orders""","""order_delivered_customer_date""",99441,96476,2965,2.98
"""products""","""product_photos_qty""",32951,32341,610,1.85
"""products""","""product_description_lenght""",32951,32341,610,1.85
"""products""","""product_name_lenght""",32951,32341,610,1.85
"""products""","""product_category_name""",32951,32341,610,1.85
"""orders""","""order_delivered_carrier_date""",99441,97658,1783,1.79
"""orders""","""order_approved_at""",99441,99281,160,0.16


In [8]:
pl.read_database(
        query = """
        select count(*)
        from raw.products
        where product_category_name is null;
        """, connection = engine
        )

count
i64
610


In [9]:
pl.read_database(
        query = """
        select count(*)
        from raw.products
        where product_category_name is null
        and product_name_lenght is null
        and product_description_lenght is null
        and product_photos_qty is null;
        """, connection = engine
        )

count
i64
610


In [19]:
pl.read_database(
        query = """
        select count(distinct oi.product_id)
        from raw.order_items oi 
        join raw.products p 
        on oi.product_id = p.product_id 
        where p.product_category_name is null;
        """, connection = engine
        )

count
i64
610


In [20]:
pl.read_database(
        query = """
        select count(distinct seller_id)
        from raw.order_items oi 
        join raw.products p 
        on oi.product_id = p.product_id 
        where p.product_category_name is null
        """, connection = engine
        )

count
i64
257


In [12]:
pl.read_database(
        query = """
        select order_status,
        count(*)
        from raw.orders 
        where order_delivered_customer_date is null
        group by order_status 
        order by count(*) desc;
        """, connection = engine
        )

order_status,count
str,i64
"""shipped""",1107
"""canceled""",619
"""unavailable""",609
"""invoiced""",314
"""processing""",301
"""delivered""",8
"""created""",5
"""approved""",2


In [13]:
pl.read_database(
        query = """
        select order_status,
        count(*)
        from raw.orders 
        where order_delivered_carrier_date is null
        group by order_status 
        order by count(*) desc;
        """, connection = engine
        )

order_status,count
str,i64
"""unavailable""",609
"""canceled""",550
"""invoiced""",314
"""processing""",301
"""created""",5
"""approved""",2
"""delivered""",2


In [14]:
pl.read_database(
        query = """
        select order_status,
        count(*)
        from raw.orders 
        where order_approved_at is null
        group by order_status 
        order by count(*) desc;
        """, connection = engine
        )

order_status,count
str,i64
"""canceled""",141
"""delivered""",14
"""created""",5


In [15]:
pl.read_database(
        query = """
        select *
        from raw.orders 
        where order_approved_at is null
        limit 20;
        """, connection = engine
        )

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,str,null,str,str,str
"""00b1cb0320190ca0daa2c88b35206009""","""3532ba38a3fd242259a514ac2b6ae6b6""","""canceled""","""2018-08-28 15:26:39""",null,null,null,"""2018-09-12 00:00:00"""
"""ed3efbd3a87bea76c2812c66a0b32219""","""191984a8ba4cbb2145acb4fe35b69664""","""canceled""","""2018-09-20 13:54:16""",null,null,null,"""2018-10-17 00:00:00"""
"""df8282afe61008dc26c6c31011474d02""","""aa797b187b5466bc6925aaaa4bb3bed1""","""canceled""","""2017-03-04 12:14:30""",null,null,null,"""2017-04-10 00:00:00"""
"""8d4c637f1accf7a88a4555f02741e606""","""b1dd715db389a2077f43174e7a675d07""","""canceled""","""2018-08-29 16:27:49""",null,null,null,"""2018-09-13 00:00:00"""
"""7a9d4c7f9b068337875b95465330f2fc""","""7f71ae48074c0cfec9195f88fcbfac55""","""canceled""","""2017-05-01 16:12:39""",null,null,null,"""2017-05-30 00:00:00"""
"""ddaec6fff982b13e7e048b627a11d6da""","""68f4ad79cc0c2ad06e19088f5c00e9fa""","""canceled""","""2016-10-04 19:41:32""",null,null,null,"""2016-11-16 00:00:00"""
"""5290c34bd38a8a095b885f13958db1e1""","""92af427e290117f39d9ff908566072e0""","""canceled""","""2018-08-21 10:25:18""",null,null,null,"""2018-09-06 00:00:00"""
"""03310aa823a66056268a3bab36e827fb""","""25dbbf0c477fd4ae0880aaffbb12e8b3""","""canceled""","""2018-08-07 16:33:59""",null,null,null,"""2018-09-04 00:00:00"""
"""4c8b9947280829d0a8b7e81cc249b875""","""403c35c4d8813bf67b3d396b91ca1619""","""canceled""","""2018-08-09 14:54:47""",null,null,null,"""2018-08-21 00:00:00"""


In [21]:
pl.read_database(
        query = """
        select *
        from raw.orders 
        where order_status = 'delivered'
        and order_approved_at is null;
        """, connection = engine
        )

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,str,null,str,str,str
"""e04abd8149ef81b95221e88f6ed9ab6a""","""2127dc6603ac33544953ef05ec155771""","""delivered""","""2017-02-18 14:40:00""",null,"""2017-02-23 12:04:47""","""2017-03-01 13:25:33""","""2017-03-17 00:00:00"""
"""8a9adc69528e1001fc68dd0aaebbb54a""","""4c1ccc74e00993733742a3c786dc3c1f""","""delivered""","""2017-02-18 12:45:31""",null,"""2017-02-23 09:01:52""","""2017-03-02 10:05:06""","""2017-03-21 00:00:00"""
"""7013bcfc1c97fe719a7b5e05e61c12db""","""2941af76d38100e0f8740a374f1a5dc3""","""delivered""","""2017-02-18 13:29:47""",null,"""2017-02-22 16:25:25""","""2017-03-01 08:07:38""","""2017-03-17 00:00:00"""
"""5cf925b116421afa85ee25e99b4c34fb""","""29c35fc91fc13fb5073c8f30505d860d""","""delivered""","""2017-02-18 16:48:35""",null,"""2017-02-22 11:23:10""","""2017-03-09 07:28:47""","""2017-03-31 00:00:00"""
"""12a95a3c06dbaec84bcfb0e2da5d228a""","""1e101e0daffaddce8159d25a8e53f2b2""","""delivered""","""2017-02-17 13:05:55""",null,"""2017-02-22 11:23:11""","""2017-03-02 11:09:19""","""2017-03-20 00:00:00"""
"""c1d4211b3dae76144deccd6c74144a88""","""684cb238dc5b5d6366244e0e0776b450""","""delivered""","""2017-01-19 12:48:08""",null,"""2017-01-25 14:56:50""","""2017-01-30 18:16:01""","""2017-03-01 00:00:00"""
"""d69e5d356402adc8cf17e08b5033acfb""","""68d081753ad4fe22fc4d410a9eb1ca01""","""delivered""","""2017-02-19 01:28:47""",null,"""2017-02-23 03:11:48""","""2017-03-02 03:41:58""","""2017-03-27 00:00:00"""
"""d77031d6a3c8a52f019764e68f211c69""","""0bf35cac6cc7327065da879e2d90fae8""","""delivered""","""2017-02-18 11:04:19""",null,"""2017-02-23 07:23:36""","""2017-03-02 16:15:23""","""2017-03-22 00:00:00"""
"""7002a78c79c519ac54022d4f8a65e6e8""","""d5de688c321096d15508faae67a27051""","""delivered""","""2017-01-19 22:26:59""",null,"""2017-01-27 11:08:05""","""2017-02-06 14:22:19""","""2017-03-16 00:00:00"""


In [22]:
pl.read_database(
        query = """
        select date_trunc('month', order_purchase_timestamp::timestamp) as month,
        count(*)
        from raw.orders 
        where order_approved_at is null
        and order_status = 'delivered'
        group by 1
        order by 1;
        """, connection = engine
        )

month,count
datetime[μs],i64
2017-01-01 00:00:00,2
2017-02-01 00:00:00,12


In [2]:
pl.read_database(
        query = """
        select count(*)
        from raw.order_items oi 
        left join raw.orders o
        on oi.order_id = o.order_id
        where o.order_id is null;
        """, connection = engine
        )

count
i64
0


In [3]:
pl.read_database(
        query = """
        select count(*)
        from raw.order_items o 
        left join raw.products p
        on o.product_id = p.product_id
        where p.product_id is null;
        """, connection = engine
        )

count
i64
0


In [4]:
pl.read_database(
        query = """
        select count(*)
        from raw.order_items o 
        left join raw.sellers s 
        on o.seller_id = s.seller_id
        where s.seller_id is null;
        """, connection = engine
        )

count
i64
0


In [5]:
pl.read_database(
        query = """
        select count(*)
        from raw.orders o 
        left join raw.customers c 
        on o.customer_id = c.customer_id
        where c.customer_id is null;
        """, connection = engine
        )

count
i64
0


In [6]:
pl.read_database(
        query = """
        select count(*)
        from raw.payments p 
        left join raw.orders o 
        on p.order_id = o.order_id
        where o.order_id is null;
        """, connection = engine
        )

count
i64
0


In [7]:
pl.read_database(
        query = """
        select count(*)
        from raw.reviews r 
        left join raw.orders o 
        on r.order_id = o.order_id
        where o.order_id is null;
        """, connection = engine
        )

count
i64
0


In [9]:
pl.read_database(
        query = """
        select column_name
        from information_schema.columns 
        where table_schema = 'raw'
        and table_name = 'orders';
        """, connection = engine
        )

column_name
str
"""order_id"""
"""customer_id"""
"""order_status"""
"""order_purchase_timestamp"""
"""order_approved_at"""
"""order_delivered_carrier_date"""
"""order_delivered_customer_date"""
"""order_estimated_delivery_date"""


In [11]:
pl.read_database(
        query = """
        select order_id
        from raw.orders
        where order_approved_at is not null
        and order_purchase_timestamp::timestamp >
        order_approved_at::timestamp;
        """, connection = engine
        )

order_id
null


In [21]:
pl.read_database(
        query = """
        select *
        from raw.orders
        where order_approved_at is not null
        and order_approved_at::timestamp >
        order_delivered_carrier_date::timestamp;
        """, connection = engine
        )

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,str,str,str,str,str
"""dcb36b511fcac050b97cd5c05de84dc3""","""3b6828a50ffe546942b7a473d70ac0fc""","""delivered""","""2018-06-07 19:03:12""","""2018-06-12 23:31:02""","""2018-06-11 14:54:00""","""2018-06-21 15:34:32""","""2018-07-04 00:00:00"""
"""688052146432ef8253587b930b01a06d""","""81e08b08e5ed4472008030d70327c71f""","""delivered""","""2018-04-22 08:48:13""","""2018-04-24 18:25:22""","""2018-04-23 19:19:14""","""2018-04-24 19:31:58""","""2018-05-15 00:00:00"""
"""58d4c4747ee059eeeb865b349b41f53a""","""1755fad7863475346bc6c3773fe055d3""","""delivered""","""2018-07-21 12:49:32""","""2018-07-26 23:31:53""","""2018-07-24 12:57:00""","""2018-07-25 23:58:19""","""2018-07-31 00:00:00"""
"""412fccb2b44a99b36714bca3fef8ad7b""","""c6865c523687cb3f235aa599afef1710""","""delivered""","""2018-07-22 22:30:05""","""2018-07-23 12:31:53""","""2018-07-23 12:24:00""","""2018-07-24 19:26:42""","""2018-07-31 00:00:00"""
"""56a4ac10a4a8f2ba7693523bb439eede""","""78438ba6ace7d2cb023dbbc81b083562""","""delivered""","""2018-07-22 13:04:47""","""2018-07-27 23:31:09""","""2018-07-24 14:03:00""","""2018-07-28 00:05:39""","""2018-08-06 00:00:00"""
"""32e4fa9bb468884309b58b9348de70c3""","""e54367d4b00c5cb76d2dfe71b9bdb89c""","""delivered""","""2018-07-04 16:49:21""","""2018-07-05 16:33:06""","""2018-07-05 14:50:00""","""2018-07-07 14:41:18""","""2018-07-23 00:00:00"""
"""4df92d82d79c3b52c7138679fa9b07fc""","""ba0660bf3fffe505ee892e153a2fbd49""","""delivered""","""2018-07-24 11:32:11""","""2018-07-29 23:30:52""","""2018-07-26 14:46:00""","""2018-07-27 18:55:57""","""2018-08-06 00:00:00"""
"""16e38caa92e342c7780f68832f832d4d""","""d29935fdaf4a76f34653da3def4f2a24""","""delivered""","""2018-05-07 01:09:09""","""2018-05-07 16:52:39""","""2018-05-07 15:09:00""","""2018-05-24 00:31:18""","""2018-06-07 00:00:00"""
"""b9afddbdcfadc9a87b41a83271c3e888""","""85c6af75161b8b2b1af98e82b5a6a5a5""","""delivered""","""2018-08-16 13:50:48""","""2018-08-16 14:05:13""","""2018-08-16 13:27:00""","""2018-08-24 14:58:37""","""2018-09-04 00:00:00"""


In [27]:
pl.read_database(
        query = """
        with anamolies as (
        select *
        from raw.orders
        where order_approved_at is not null
        and order_approved_at::timestamp >
        order_delivered_carrier_date::timestamp )

        select order_id,
        order_purchase_timestamp,
        order_approved_at,
        order_delivered_carrier_date,
        order_delivered_customer_date
        from anamolies;
        """, connection = engine
        )

order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
str,str,str,str,str
"""dcb36b511fcac050b97cd5c05de84dc3""","""2018-06-07 19:03:12""","""2018-06-12 23:31:02""","""2018-06-11 14:54:00""","""2018-06-21 15:34:32"""
"""688052146432ef8253587b930b01a06d""","""2018-04-22 08:48:13""","""2018-04-24 18:25:22""","""2018-04-23 19:19:14""","""2018-04-24 19:31:58"""
"""58d4c4747ee059eeeb865b349b41f53a""","""2018-07-21 12:49:32""","""2018-07-26 23:31:53""","""2018-07-24 12:57:00""","""2018-07-25 23:58:19"""
"""412fccb2b44a99b36714bca3fef8ad7b""","""2018-07-22 22:30:05""","""2018-07-23 12:31:53""","""2018-07-23 12:24:00""","""2018-07-24 19:26:42"""
"""56a4ac10a4a8f2ba7693523bb439eede""","""2018-07-22 13:04:47""","""2018-07-27 23:31:09""","""2018-07-24 14:03:00""","""2018-07-28 00:05:39"""
"""32e4fa9bb468884309b58b9348de70c3""","""2018-07-04 16:49:21""","""2018-07-05 16:33:06""","""2018-07-05 14:50:00""","""2018-07-07 14:41:18"""
"""4df92d82d79c3b52c7138679fa9b07fc""","""2018-07-24 11:32:11""","""2018-07-29 23:30:52""","""2018-07-26 14:46:00""","""2018-07-27 18:55:57"""
"""16e38caa92e342c7780f68832f832d4d""","""2018-05-07 01:09:09""","""2018-05-07 16:52:39""","""2018-05-07 15:09:00""","""2018-05-24 00:31:18"""
"""b9afddbdcfadc9a87b41a83271c3e888""","""2018-08-16 13:50:48""","""2018-08-16 14:05:13""","""2018-08-16 13:27:00""","""2018-08-24 14:58:37"""


In [18]:
pl.read_database(
        query = """
        select order_id
        from raw.orders
        where order_approved_at is not null
        and order_delivered_carrier_date::timestamp >
        order_delivered_customer_date::timestamp;
        """, connection = engine
        )

order_id
str
"""a1abeb653a4d4cd1e142ccb8c82cd069"""
"""383aa8b2724fe452d9ccd9934a8c628b"""
"""cb1134f9010d242e9515ad1c78ec0c39"""
"""dceb62e8fa94b46006c9554fed743df0"""
"""5f9d46795c3126674e52becb3a1a517f"""
"""8c78d01de3a9009e23d6877a7cc9be20"""
"""b27af682321527a6349f1761eb3f360c"""
"""1cc3ae63caffff2d6c3ee3e78e074acf"""
"""e37f11cae9985ca58f0b56f268720537"""


In [20]:
pl.read_database(
        query = """
        select order_id
        from raw.orders
        where order_approved_at is not null
        and order_delivered_customer_date is not null
        and order_approved_at::timestamp >
        order_delivered_customer_date::timestamp;
        """, connection = engine
        )

order_id
str
"""58d4c4747ee059eeeb865b349b41f53a"""
"""4df92d82d79c3b52c7138679fa9b07fc"""
"""6e57e23ecac1ae881286657694444267"""
"""f222c56f035b47dfa1e069a88235d730"""
"""cf72398d0690f841271b695bbfda82d2"""
"""6df6c9c9af6ef75b4f06f8a7b9f47e9c"""
"""1fab4ac9d85079b3da72a11475ae1685"""
"""fa962e76e50f3469ae2abfa54e6d5be0"""
"""0467205a89711e4ec8e70ef2277e3287"""


In [29]:
pl.read_database(
        query = """
        select count(*)
        from (
            select review_id
            from raw.reviews
            group by review_id 
            having count(*) > 1
            ) t;
        """, connection = engine
        )

count
i64
789


In [31]:
pl.read_database(
        query = """
        select count(*)
        from (
            select order_id
            from raw.reviews
            group by order_id 
            having count(*) > 1
            ) t;
        """, connection = engine
        )

count
i64
547


In [32]:
pl.read_database(
        query = """
        select review_id
        from raw.reviews
        group by review_id
        having count(*) > 1;
        """, connection = engine
        )

review_id
str
"""4fea089659b78b233489192ca88d8448"""
"""e9cf1b5368603de7222e94123cc1ceb5"""
"""f9306ef54d4adecf8cca0377b9b0f957"""
"""22a299893c5cee48be41d8fab8804e6e"""
"""da8e936bc2f1b52ec626f8aa41f3df44"""
"""d8949cc65047d1fb816ac8c2500ad4b1"""
"""393907d8e4b794865eff74489af712aa"""
"""37680aaaa05c4feca6dc54850acc4bda"""
"""3d635999da844a3b3e463959a0aff863"""


In [34]:
pl.read_database(
        query = """
        select * 
        from raw.reviews
        where review_id = '4fea089659b78b233489192ca88d8448';
        """, connection = engine
        )

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
str,str,i64,null,null,str,str
"""4fea089659b78b233489192ca88d8448""","""3200257acc9dd36d1fa819e72fca7e59""",4,null,null,"""2017-12-05 00:00:00""","""2017-12-08 00:30:08"""
"""4fea089659b78b233489192ca88d8448""","""6390d8596b48fe36ecbd508af3770dec""",4,null,null,"""2017-12-05 00:00:00""","""2017-12-08 00:30:08"""


In [35]:
pl.read_database(
        query = """
        select *
        from raw.orders
        where order_id in (
            '3200257acc9dd36d1fa819e72fca7e59',
            '6390d8596b48fe36ecbd508af3770dec');
        """, connection = engine
        )

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
str,str,str,str,str,str,str,str
"""3200257acc9dd36d1fa819e72fca7e59""","""c8a84c4b2aea91e2cfd48a87ae2a8947""","""delivered""","""2017-11-24 12:58:53""","""2017-11-24 15:13:44""","""2017-11-28 15:29:05""","""2017-12-04 20:33:01""","""2017-12-27 00:00:00"""
"""6390d8596b48fe36ecbd508af3770dec""","""5f302af18bb0da175bc7b0eff5307451""","""delivered""","""2017-11-24 12:58:54""","""2017-11-24 15:13:32""","""2017-11-27 21:48:39""","""2017-12-05 00:32:49""","""2017-12-28 00:00:00"""


In [36]:
pl.read_database(
        query = """
        select *
        from raw.order_items
        where order_id in (
            '3200257acc9dd36d1fa819e72fca7e59',
            '6390d8596b48fe36ecbd508af3770dec');
        """, connection = engine
        )

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,str,f64,f64
"""3200257acc9dd36d1fa819e72fca7e59""",1,"""167b4b8c4bd0c401bea62f5e050d70a4""","""25c5c91f63607446a97b143d2d535d31""","""2017-12-11 15:13:44""",149.87,16.81
"""3200257acc9dd36d1fa819e72fca7e59""",2,"""167b4b8c4bd0c401bea62f5e050d70a4""","""25c5c91f63607446a97b143d2d535d31""","""2017-12-11 15:13:44""",149.87,16.81
"""6390d8596b48fe36ecbd508af3770dec""",1,"""e46d3c8d60e3f8b382b6f239e5af08ac""","""1c68394e931a64f90ea236c5ea590300""","""2017-12-07 15:13:32""",679.12,16.88


In [37]:
pl.read_database(
        query = """
        select order_id
        from raw.reviews
        group by order_id
        having count(*) > 1;
        """, connection = engine
        )

order_id
str
"""565b0bdb5bfef65df5a23890967586f6"""
"""6318d8c5f6857de384c0b3186d036457"""
"""b9a528ec4ad891d009b3dc50588218f1"""
"""6b2f7dba402cf63ca207bbe2cc05ef67"""
"""fa350da9efbba83c2957b629bf6c2c7c"""
"""61b2314bd576f3d792e46e1d56e51ee6"""
"""95e7f49dc56e12097c265c45527a3941"""
"""a89773db642a632c5cf8534ce1a49b28"""
"""95287dc5df93f25b90309d5acddbe02e"""


In [38]:
pl.read_database(
        query = """
        select *
        from raw.reviews
        where order_id = '565b0bdb5bfef65df5a23890967586f6';
        """, connection = engine
        )

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
str,str,i64,null,null,str,str
"""b5170254268050e30337c3f408805395""","""565b0bdb5bfef65df5a23890967586f6""",5,null,null,"""2017-09-05 00:00:00""","""2017-09-08 23:08:53"""
"""9487a43f29708b00c4b6ef5af2441d1f""","""565b0bdb5bfef65df5a23890967586f6""",1,null,null,"""2017-09-07 00:00:00""","""2017-09-13 09:52:05"""


In [39]:
pl.read_database(
        query = """
        select *
        from raw.products
        where product_weight_g is null;
        """, connection = engine
        )

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
str,str,i64,i64,i64,null,null,null,null
"""09ff539a621711667c43eba6a3bd8466""","""bebes""",60,865,3,null,null,null,null
"""5eb564652db742ff8f28759cd8d2652a""",null,null,null,null,null,null,null,null


In [42]:
pl.read_database(
        query = """
        select count(*) as total_rows,
        count(distinct geolocation_zip_code_prefix) as unique_zips
        from raw.geolocation;
        """, connection = engine
        )

total_rows,unique_zips
i64,i64
1000163,19015


In [45]:
pl.read_database(
        query = """
        select *
        from raw.geolocation
        where geolocation_zip_code_prefix = (
            select geolocation_zip_code_prefix
            from raw.geolocation
            group by 1
            having count(*) > 50
            limit 1)
        limit 20;
        """, connection = engine
        )

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
i64,f64,f64,str,str
1014,-23.546435,-46.63383,"""sao paulo""","""SP"""
1014,-23.547082,-46.633408,"""sao paulo""","""SP"""
1014,-23.546453,-46.633947,"""sao paulo""","""SP"""
1014,-23.546319,-46.633431,"""sao paulo""","""SP"""
1014,-23.546941,-46.633312,"""sao paulo""","""SP"""
1014,-23.546762,-46.633455,"""sao paulo""","""SP"""
1014,-23.545338,-46.633582,"""sao paulo""","""SP"""
1014,-23.547216,-46.633386,"""sao paulo""","""SP"""
1014,-23.545949,-46.633501,"""são paulo""","""SP"""
